# 🛰️ Earth AI Remote Sensing: DIOR Object Detection

This notebook provides an end-to-end object detection example on the **[DIOR]** dataset using a pretrained **GeoFM ViT-Large Patch16** backbone with **Faster R-CNN**.

It demonstrates:

* Downloading and preparing the DIOR dataset
* Converting DIOR XML annotations to YOLO format
* Building a multi-scale GeoFM feature pyramid
* Fine-tuning Faster R-CNN with separate encoder and detector learning rates

In [ ]:
!pip install git+https://github.com/google-research/remote-sensing.git

In [ ]:
!pip install -q torchmetrics faster-coco-eval

In [ ]:
import os
from pathlib import Path
import random
import subprocess
import zipfile

# Custom dataset, data loader, and class names
from dior_dataset import CLASS_NAMES, DIORYOLODataset, collate_fn

# Pretrained remote-sensing Vision Transformer backbone and architecture
from remote_sensing.models import vits
from remote_sensing.models.architectures import ViTBackbonePyramid
from remote_sensing.models.visualizations import (
    ObjectDetectionVisualizationConfig,
    visualize_dataset_predictions,
    visualize_object_detection,
)

# Core PyTorch components & detection utilities
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from tqdm.auto import tqdm

# The encoder uses a smaller learning rate to preserve its pretrained features.
LEARNING_RATE = 1e-3
ENCODER_LEARNING_RATE = 2e-6

# All images and annotations are resized to this spatial resolution.
IMAGE_SIZE = 800

# DIOR contains 20 foreground categories.
# Torchvision detection models reserve class index 0 for the background.
NUM_CLASSES = 21

BATCH_SIZE = 16
NUM_EPOCHS = 200

# Multiprocessing is disabled for better notebook compatibility.
# This can be increased when running the code as a standalone script.
NUM_WORKERS = 0


# Pretrained GeoFM checkpoint
MODEL_NAME = "GeoFM_Object_Detection_VitLarge_Patch16_RGB_V2"

MODEL_GCS = f"gs://path/to/models/{MODEL_NAME}"


# Expected local checkpoint location.
LOCAL_MODEL_PATH = f"./models/{MODEL_NAME}"
GEOFM_CKPT = f"{LOCAL_MODEL_PATH}/torch"

# 📦 Model and Dataset Setup

This section prepares the pretrained GeoFM checkpoint and the DIOR dataset.

* Downloads the GeoFM ViT-Large object detection checkpoint from Google Cloud Storage.
* Downloads and extracts the DIOR dataset archives.
* Runs `preprocess_dior.py` to organize the dataset and convert XML annotations into YOLO-format labels.

In [ ]:
# Download GeoFM Object Detection backbone

if not os.path.exists(GEOFM_CKPT):
  print("Copying GeoFM model from GCS")
  os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)
  !gcloud storage cp --recursive {MODEL_GCS}/* {LOCAL_MODEL_PATH}/
else:
  print("GeoFM model already exists")

print("GeoFM checkpoint:", GEOFM_CKPT)
print(os.listdir(GEOFM_CKPT))

assert os.path.exists(f"{GEOFM_CKPT}/config.json")
assert os.path.exists(f"{GEOFM_CKPT}/pytorch_model.bin") or os.path.exists(
    f"{GEOFM_CKPT}/model.safetensors"
)

print("GeoFM checkpoint ready")

In [ ]:
# Download and extract the DIOR dataset
LOCAL_DATASET_PATH = Path("./datasets/DIOR")
ARCHIVE_PATH = LOCAL_DATASET_PATH / "archives"

LOCAL_DATASET_PATH.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH.mkdir(parents=True, exist_ok=True)

DATASET_GCS = "gs://path/to/dior-dataset"
# Please download DIOR dataset archives from https://drive.google.com/open?id=1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC and place them in ./datasets/DIOR/archives


# Extract all archives into the local DIOR directory.
! unzip -q {ARCHIVE_PATH}/*.zip -d {LOCAL_DATASET_PATH}

print(f"DIOR dataset available at: {LOCAL_DATASET_PATH.resolve()}")

In [ ]:
!python preprocess_dior.py

In [ ]:
# Dataset and DataLoader initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

LOCAL_DATASET_PATH = Path("datasets/DIOR")

train_image_dir = LOCAL_DATASET_PATH / "train" / "images"
train_label_dir = LOCAL_DATASET_PATH / "train" / "labels"

valid_image_dir = LOCAL_DATASET_PATH / "valid" / "images"
valid_label_dir = LOCAL_DATASET_PATH / "valid" / "labels"

# Verify that preprocessing created the expected directories.
for directory in [
    train_image_dir,
    train_label_dir,
    valid_image_dir,
    valid_label_dir,
]:
  if not directory.exists():
    raise FileNotFoundError(f"Required directory not found: {directory}")

trainval_ds = DIORYOLODataset(
    img_dir=train_image_dir,
    label_dir=train_label_dir,
    image_size=IMAGE_SIZE,
    train=True,
    hflip_prob=0.5,
)

test_ds = DIORYOLODataset(
    img_dir=valid_image_dir,
    label_dir=valid_label_dir,
    image_size=IMAGE_SIZE,
    train=False,
)

train_loader = DataLoader(
    trainval_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=(device.type == "cuda"),
)

print("Training directory:", trainval_ds.img_dir)
print("Training samples:", len(trainval_ds))
print("Validation directory:", test_ds.img_dir)
print("Validation samples:", len(test_ds))

# Test that one sample can be loaded successfully.
sample_image, sample_target = trainval_ds[0]

print("Sample image shape:", sample_image.shape)
print("Sample boxes shape:", sample_target["boxes"].shape)

In [ ]:
# Visualize sample ground truth annotations
config = ObjectDetectionVisualizationConfig(
    class_names=CLASS_NAMES,
    score_threshold=0.5,
)

fig = visualize_object_detection(
    images=[sample_image],
    pred_boxes=[sample_target["boxes"]],
    pred_classes=[sample_target["labels"]],
    pred_scores=[
        torch.ones_like(sample_target["labels"], dtype=torch.float32)
    ],
    config=config,
)
plt.show()

# 🧠 Faster R-CNN Model Construction

This section builds the object detector using the pretrained **GeoFM ViT-Large Patch16** backbone.

* Creates a five-level feature pyramid from the GeoFM representation.
* Assigns anchor scales from 16 to 256 pixels across pyramid levels.
* Configures multi-scale ROI pooling for object classification and box regression.
* Initializes Faster R-CNN for the 20 DIOR classes plus background.


In [ ]:
# Build Faster R-CNN with the ViT backbone
backbone = ViTBackbonePyramid(
    ckpt_path=GEOFM_CKPT,
    image_size=IMAGE_SIZE,
    out_channels=256,
    freeze_encoder=False,
)

# One anchor scale is assigned to each feature-map level.
anchor_generator = AnchorGenerator(
    sizes=(
        (16,),
        (32,),
        (64,),
        (128,),
        (256,),
    ),
    aspect_ratios=(
        (0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0),
        (0.5, 1.0, 2.0),
    ),
)

box_roi_pool = MultiScaleRoIAlign(
    featmap_names=["0", "1", "2", "3", "4"],
    output_size=7,
    sampling_ratio=2,
)

model = FasterRCNN(
    backbone=backbone,
    num_classes=NUM_CLASSES,
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=box_roi_pool,
    min_size=IMAGE_SIZE,
    max_size=IMAGE_SIZE,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)

model = model.to(device)

print(
    "Faster R-CNN with GeoFM ViT-Large Patch16 "
    "single-feature pyramid backbone is ready."
)

In [ ]:
# Optimizer with linear warmup and cosine LR decay
WEIGHT_DECAY = 1e-4
NUM_WARMUP_EPOCHS = 10

model = model.to(device)

# Use a lower learning rate for the pretrained GeoFM encoder.
encoder_params = [
    parameter
    for parameter in model.backbone.encoder.parameters()
    if parameter.requires_grad
]

encoder_param_ids = {id(parameter) for parameter in encoder_params}

# All remaining trainable parameters belong to the feature projection,
# RPN, ROI pooling, classification head, and box-regression head.
other_params = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad and id(parameter) not in encoder_param_ids
]

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_params,
            "lr": ENCODER_LEARNING_RATE,
            "name": "encoder",
        },
        {
            "params": other_params,
            "lr": LEARNING_RATE,
            "name": "detector",
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

# The scheduler is updated once per training batch.
steps_per_epoch = len(train_loader)
num_warmup_steps = NUM_WARMUP_EPOCHS * steps_per_epoch
total_steps = NUM_EPOCHS * steps_per_epoch

if not 0 < num_warmup_steps < total_steps:
  raise ValueError(
      "Warmup steps must be between 0 and total steps, but received "
      f"{num_warmup_steps} warmup steps and {total_steps} total steps."
  )

# Increase both learning rates linearly from 1% to 100%
# of their configured values.
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=num_warmup_steps,
)

# Decay both parameter groups using the same multiplicative cosine curve,
# preserving the relative LR difference between encoder and detector.
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps - num_warmup_steps,
    eta_min=0.0,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        warmup_scheduler,
        cosine_scheduler,
    ],
    milestones=[num_warmup_steps],
)

global_step = 0

print("Model device:", next(model.parameters()).device)
print("Encoder parameters:", sum(p.numel() for p in encoder_params))
print("Detector/head parameters:", sum(p.numel() for p in other_params))
print("Encoder LR:", ENCODER_LEARNING_RATE)
print("Detector LR:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Warmup epochs:", NUM_WARMUP_EPOCHS)
print("Warmup steps:", num_warmup_steps)
print("Total optimization steps:", total_steps)

# 🚀 Model Training

This section fine-tunes Faster R-CNN on DIOR using mixed-precision training.

* Computes the combined detection loss for each batch.
* Applies gradient clipping for stable optimization.
* Updates the warmup and cosine learning-rate schedule after each step.
* Skips non-finite losses and reports training progress.

In [ ]:
# Training loop

SAVE_DIR = Path("./output_fasterrcnn")
SAVE_EVERY_EPOCHS = 1
MAX_GRAD_NORM = 1.0

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# bfloat16 reduces memory usage without requiring gradient scaling.
use_autocast = device.type == "cuda"

for epoch in range(NUM_EPOCHS):
  model.train()

  epoch_loss = 0.0
  successful_steps = 0
  skipped_steps = 0

  progress_bar = tqdm(
      train_loader,
      desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
      leave=False,
      mininterval=10,
  )

  for step, (images, targets) in enumerate(progress_bar):
    # Torchvision detection models expect lists of images and targets.
    images = [image.to(device, non_blocking=True) for image in images]

    targets = [
        {
            key: value.to(device, non_blocking=True)
            for key, value in target.items()
        }
        for target in targets
    ]

    optimizer.zero_grad(set_to_none=True)

    # Faster R-CNN returns its individual training losses as a dictionary.
    with torch.autocast(
        device_type=device.type,
        dtype=torch.bfloat16,
        enabled=use_autocast,
    ):
      loss_dict = model(images, targets)
      loss = sum(loss_dict.values())

    # Skip invalid optimization steps without advancing the LR scheduler.
    if not torch.isfinite(loss):
      skipped_steps += 1

      print(
          f"\nNon-finite loss at epoch {epoch + 1}, step {step}: {loss.item()}"
      )

      print({
          name: float(value.detach().cpu()) for name, value in loss_dict.items()
      })

      continue

    loss.backward()

    # Limit unusually large updates during detector fine-tuning.
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=MAX_GRAD_NORM,
    )

    optimizer.step()

    # The scheduler is configured in optimization steps rather than epochs.
    scheduler.step()
    global_step += 1

    loss_value = float(loss.detach().cpu())

    epoch_loss += loss_value
    successful_steps += 1

    # Update the progress display periodically to reduce notebook overhead.
    if step % 100 == 0:
      detector_lr = optimizer.param_groups[1]["lr"]
      encoder_lr = optimizer.param_groups[0]["lr"]

      progress_bar.set_postfix(
          loss=f"{loss_value:.4f}",
          lr=f"{detector_lr:.2e}",
          enc_lr=f"{encoder_lr:.2e}",
          grad=f"{float(gradient_norm):.2f}",
      )

  if successful_steps == 0:
    raise RuntimeError(
        f"Epoch {epoch + 1} completed without a valid optimization step."
    )

  average_loss = epoch_loss / successful_steps

  print(
      f"Epoch {epoch + 1}/{NUM_EPOCHS}: "
      f"loss={average_loss:.4f}, "
      f"lr={optimizer.param_groups[1]['lr']:.2e}, "
      f"enc_lr={optimizer.param_groups[0]['lr']:.2e}, "
      f"skipped={skipped_steps}"
  )

  # Save all state required to resume training.
  if (epoch + 1) % SAVE_EVERY_EPOCHS == 0:
    checkpoint = {
        "epoch": epoch + 1,
        "global_step": global_step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "average_loss": average_loss,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "learning_rate": LEARNING_RATE,
        "encoder_learning_rate": ENCODER_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": NUM_WARMUP_EPOCHS,
    }

    checkpoint_path = (
        SAVE_DIR / f"fasterrcnn_geofm_patch16_pyramid_epoch_{epoch + 1}.pth"
    )

    torch.save(checkpoint, checkpoint_path)

    print(f"Checkpoint saved: {checkpoint_path}")

# 🔍 Inference Visualization

This section runs the trained detector on randomly selected test images and displays the predicted bounding boxes, class labels, and confidence scores side by side.


In [ ]:
config = ObjectDetectionVisualizationConfig(
    class_names=CLASS_NAMES,
    score_threshold=0.5,
)

fig = visualize_dataset_predictions(
    model=model,
    dataset=test_ds,
    config=config,
    device=device,
    num_images=4,
    seed=42,
)
plt.show()